# PatentsView — Data Profiling
We profile each raw TSV file before writing any cleaning code.
Large files are read in chunks to avoid blowing up RAM.
A 250-row sample is pulled separately for eyeballing actual values.

In [2]:
import csv
import pandas as pd
import os

RAW_DIR = '../data/raw'
ENCODING = 'latin-1'
CHUNK_SIZE = 100_000

CHUNK_FILES = {
    'g_patent.tsv',
    'g_patent_abstract.tsv',
    'g_application.tsv',
    'g_assignee_disambiguated.tsv',
    'g_inventor_disambiguated.tsv',
}

# g_patent_abstract.tsv has two compounding problems:
#   1. Some abstracts are so long they overflow the C parser's internal buffer
#   2. Abstract text contains unescaped quote characters that confuse the parser
# The only combination that handles both is: python engine + QUOTE_NONE
PROBLEM_FILES = {
    'g_patent_abstract.tsv',
}


def make_reader(path, filename, chunksize=None, nrows=None):
    """
    Single place where all read_csv parameters live.
    Problem files get the python engine and QUOTE_NONE.
    Everything else uses the faster C engine with bad line skipping.
    """
    if filename in PROBLEM_FILES:
        return pd.read_csv(
            path,
            sep='\t',
            dtype=str,
            encoding=ENCODING,
            keep_default_na=True,
            on_bad_lines='skip',
            engine='python',         # no buffer size limit
            quoting=csv.QUOTE_NONE,  # treat quotes as plain text
            chunksize=chunksize,
            nrows=nrows
        )
    else:
        return pd.read_csv(
            path,
            sep='\t',
            dtype=str,
            encoding=ENCODING,
            keep_default_na=True,
            on_bad_lines='skip',
            engine='c',
            chunksize=chunksize,
            nrows=nrows
        )


def get_sample(filename, n=250):
    """Pulls n rows for visual inspection of actual data values."""
    return make_reader(os.path.join(RAW_DIR, filename), filename, nrows=n)


def profile_chunked(filename):
    """
    Iterates through the file in chunks accumulating:
      - total row count
      - null counts per column
      - value distributions for low-cardinality columns (<=50 unique values)
    Gives the same picture as loading the full file at a fraction of RAM cost.
    """
    reader = make_reader(
        os.path.join(RAW_DIR, filename),
        filename,
        chunksize=CHUNK_SIZE
    )

    total_rows = 0
    null_counts = None
    value_counts = {}
    unique_vals = {}
    columns = None

    for i, chunk in enumerate(reader):
        if columns is None:
            columns = list(chunk.columns)
            null_counts = pd.Series(0, index=columns)
            unique_vals = {col: set() for col in columns}
            value_counts = {col: {} for col in columns}

        total_rows += len(chunk)
        null_counts += chunk.isnull().sum()

        for col in columns:
            if unique_vals.get(col) is not None:
                for val, cnt in chunk[col].value_counts(dropna=False).items():
                    value_counts[col][val] = value_counts[col].get(val, 0) + cnt
                unique_vals[col].update(chunk[col].dropna().unique())
                if len(unique_vals[col]) > 50:
                    unique_vals[col] = None

        if (i + 1) % 10 == 0:
            print(f'  ...processed {total_rows:,} rows so far')

    return total_rows, null_counts, columns, unique_vals, value_counts


def profile(filename):
    """Main profiling function — handles both small and large files."""
    print('\n' + '='*65)
    print(f'FILE: {filename}')
    print('='*65)

    if filename in CHUNK_FILES:
        note = ' (python engine + QUOTE_NONE)' if filename in PROBLEM_FILES else ''
        print(f'Large file — reading in chunks{note}...')
        total_rows, null_counts, columns, unique_vals, value_counts = profile_chunked(filename)

        print(f'\nShape: {total_rows:,} rows x {len(columns)} columns')
        print(f'Columns: {columns}')

        print('\n--- Null Counts ---')
        null_pct = (null_counts / total_rows * 100).round(2)
        print(pd.DataFrame({'null_count': null_counts, 'null_%': null_pct}).to_string())

        print('\n--- Unique Value Counts per Column ---')
        for col in columns:
            uv = unique_vals.get(col)
            label = f'{len(uv)} unique values' if uv is not None else '> 50 unique values (high cardinality)'
            print(f'  {col}: {label}')

        print('\n--- Value Distributions for Low-Cardinality Columns ---')
        for col in columns:
            uv = unique_vals.get(col)
            if uv is not None and len(uv) <= 50:
                print(f'\n  {col}:')
                print(pd.Series(value_counts[col]).sort_values(ascending=False).to_string())

    else:
        print('Small file — loading fully...')
        full = make_reader(os.path.join(RAW_DIR, filename), filename)
        total_rows = len(full)
        columns = list(full.columns)

        print(f'\nShape: {total_rows:,} rows x {len(columns)} columns')
        print(f'Columns: {columns}')

        print('\n--- Null Counts ---')
        null_counts = full.isnull().sum()
        null_pct = (null_counts / total_rows * 100).round(2)
        print(pd.DataFrame({'null_count': null_counts, 'null_%': null_pct}).to_string())

        print('\n--- Unique Value Counts per Column ---')
        for col in full.columns:
            print(f'  {col}: {full[col].nunique():,} unique values')

        print('\n--- Value Distributions for Low-Cardinality Columns ---')
        for col in full.columns:
            if full[col].nunique() <= 50:
                print(f'\n  {col}:')
                print(full[col].value_counts(dropna=False).to_string())

    print('\n--- 250-Row Sample ---')
    print(get_sample(filename).to_string())

    print(f'\nDone: {filename}')

## 1. g_patent

In [2]:
profile('g_patent.tsv')


FILE: g_patent.tsv
Large file — reading in chunks...
  ...processed 1,000,000 rows so far
  ...processed 2,000,000 rows so far
  ...processed 3,000,000 rows so far
  ...processed 4,000,000 rows so far
  ...processed 5,000,000 rows so far
  ...processed 6,000,000 rows so far
  ...processed 7,000,000 rows so far
  ...processed 8,000,000 rows so far
  ...processed 9,000,000 rows so far

Shape: 9,427,493 rows x 8 columns
Columns: ['patent_id', 'patent_type', 'patent_date', 'patent_title', 'wipo_kind', 'num_claims', 'withdrawn', 'filename']

--- Null Counts ---
              null_count  null_%
patent_id              0     0.0
patent_type          272     0.0
patent_date          272     0.0
patent_title         272     0.0
wipo_kind            274     0.0
num_claims           276     0.0
withdrawn            280     0.0
filename             281     0.0

--- Unique Value Counts per Column ---
  patent_id: > 50 unique values (high cardinality)
  patent_type: 11 unique values
  patent_date: >

## 2. g_patent_abstract
Uses python engine + QUOTE_NONE: abstracts are too long for C parser buffer
and contain unescaped quotes inside the text.

In [3]:
profile('g_patent_abstract.tsv')


FILE: g_patent_abstract.tsv
Large file — reading in chunks (python engine + QUOTE_NONE)...
  ...processed 1,000,000 rows so far
  ...processed 2,000,000 rows so far
  ...processed 3,000,000 rows so far
  ...processed 4,000,000 rows so far
  ...processed 5,000,000 rows so far
  ...processed 6,000,000 rows so far
  ...processed 7,000,000 rows so far
  ...processed 8,000,000 rows so far
  ...processed 9,000,000 rows so far

Shape: 9,876,684 rows x 2 columns
Columns: ['"patent_id"', '"patent_abstract"']

--- Null Counts ---
                   null_count  null_%
"patent_id"                 0    0.00
"patent_abstract"     1752340   17.74

--- Unique Value Counts per Column ---
  "patent_id": > 50 unique values (high cardinality)
  "patent_abstract": > 50 unique values (high cardinality)

--- Value Distributions for Low-Cardinality Columns ---

--- 250-Row Sample ---
    "patent_id"                                                                                                               

## 3. g_application

In [4]:
profile('g_application.tsv')


FILE: g_application.tsv
Large file — reading in chunks...
  ...processed 1,000,000 rows so far
  ...processed 2,000,000 rows so far
  ...processed 3,000,000 rows so far
  ...processed 4,000,000 rows so far
  ...processed 5,000,000 rows so far
  ...processed 6,000,000 rows so far
  ...processed 7,000,000 rows so far
  ...processed 8,000,000 rows so far
  ...processed 9,000,000 rows so far

Shape: 9,311,726 rows x 6 columns
Columns: ['application_id', 'patent_id', 'patent_application_type', 'filing_date', 'series_code', 'rule_47_flag']

--- Null Counts ---
                         null_count  null_%
application_id                    0    0.00
patent_id                      6987    0.08
patent_application_type        6988    0.08
filing_date                    6992    0.08
series_code                    6999    0.08
rule_47_flag                   7810    0.08

--- Unique Value Counts per Column ---
  application_id: > 50 unique values (high cardinality)
  patent_id: > 50 unique values (h

## 4. g_inventor_disambiguated

In [5]:
profile('g_inventor_disambiguated.tsv')


FILE: g_inventor_disambiguated.tsv
Large file — reading in chunks...
  ...processed 1,000,000 rows so far
  ...processed 2,000,000 rows so far
  ...processed 3,000,000 rows so far
  ...processed 4,000,000 rows so far
  ...processed 5,000,000 rows so far
  ...processed 6,000,000 rows so far
  ...processed 7,000,000 rows so far
  ...processed 8,000,000 rows so far
  ...processed 9,000,000 rows so far
  ...processed 10,000,000 rows so far
  ...processed 11,000,000 rows so far
  ...processed 12,000,000 rows so far
  ...processed 13,000,000 rows so far
  ...processed 14,000,000 rows so far
  ...processed 15,000,000 rows so far
  ...processed 16,000,000 rows so far
  ...processed 17,000,000 rows so far
  ...processed 18,000,000 rows so far
  ...processed 19,000,000 rows so far
  ...processed 20,000,000 rows so far
  ...processed 21,000,000 rows so far
  ...processed 22,000,000 rows so far
  ...processed 23,000,000 rows so far
  ...processed 24,000,000 rows so far

Shape: 24,037,380 rows x 7

## 5. g_assignee_disambiguated

In [6]:
profile('g_assignee_disambiguated.tsv')


FILE: g_assignee_disambiguated.tsv
Large file — reading in chunks...
  ...processed 1,000,000 rows so far
  ...processed 2,000,000 rows so far
  ...processed 3,000,000 rows so far
  ...processed 4,000,000 rows so far
  ...processed 5,000,000 rows so far
  ...processed 6,000,000 rows so far
  ...processed 7,000,000 rows so far
  ...processed 8,000,000 rows so far

Shape: 8,747,138 rows x 8 columns
Columns: ['patent_id', 'assignee_sequence', 'assignee_id', 'disambig_assignee_individual_name_first', 'disambig_assignee_individual_name_last', 'disambig_assignee_organization', 'assignee_type', 'location_id']

--- Null Counts ---
                                         null_count  null_%
patent_id                                         0    0.00
assignee_sequence                               305    0.00
assignee_id                                     305    0.00
disambig_assignee_individual_name_first     8588202   98.18
disambig_assignee_individual_name_last      8588184   98.18
disambig

## 6. g_location_disambiguated

In [7]:
profile('g_location_disambiguated.tsv')


FILE: g_location_disambiguated.tsv
Small file — loading fully...

Shape: 99,369 rows x 9 columns
Columns: ['location_id', 'disambig_city', 'disambig_state', 'disambig_country', 'latitude', 'longitude', 'county', 'state_fips', 'county_fips']

--- Null Counts ---
                  null_count  null_%
location_id                0    0.00
disambig_city             82    0.08
disambig_state         67731   68.16
disambig_country          10    0.01
latitude                   1    0.00
longitude                  1    0.00
county                 70456   70.90
state_fips             70448   70.90
county_fips            70456   70.90

--- Unique Value Counts per Column ---
  location_id: 99,369 unique values
  disambig_city: 84,153 unique values
  disambig_state: 70 unique values
  disambig_country: 205 unique values
  latitude: 99,263 unique values
  longitude: 99,307 unique values
  county: 3,102 unique values
  state_fips: 56 unique values
  county_fips: 318 unique values

--- Value Distribu